# The Complete Transformer Encoder-Decoder

This notebook assembles the components developed in notebook 3 into a complete sequence-to-sequence Transformer. It adds:

- a reusable English–German parallel-text dataset,
- dynamic per-batch padding and teacher-forcing shifts,
- stacked encoder and decoder blocks,
- shared source/target token embeddings and a tied output projection, and
- padding-aware next-token cross-entropy loss.

The implementation follows the post-layer-normalization structure of the original Transformer.

In [1]:
from itertools import islice, zip_longest
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tokenizers import Tokenizer

## 1. Configuration and tokenizer

The original Transformer used six encoder and six decoder layers. This notebook uses two of each by default so that a forward and backward pass remain practical on a CPU. Set `num_layers = 6` to match the paper's depth.

In [2]:
embedding_dim = 512
num_heads = 8
num_layers = 6
context_length = 128
dropout_rate = 0.1
label_smoothing = 0.1
batch_size = 4
max_examples = 10_000


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing pyproject.toml")


repo_root = find_repo_root()
tokenizer_path = repo_root / "data" / "tokenizers" / "tatoeba_bpe_en_de.json"
english_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.en"
german_path = repo_root / "data" / "raw" / "tatoeba_en_de" / "Tatoeba.de-en.de"

tokenizer = Tokenizer.from_file(str(tokenizer_path))
vocab_size = tokenizer.get_vocab_size()
pad_token_id = tokenizer.token_to_id("[PAD]")
bos_token_id = tokenizer.token_to_id("[BOS]")
eos_token_id = tokenizer.token_to_id("[EOS]")

assert None not in (pad_token_id, bos_token_id, eos_token_id)
print(f"Vocabulary size: {vocab_size:,}")
print(f"PAD/BOS/EOS ids: {pad_token_id}/{bos_token_id}/{eos_token_id}")

Vocabulary size: 16,000
PAD/BOS/EOS ids: 0/1/2


## 2. Parallel dataset and dynamic batching

The two Tatoeba files are line-aligned: English line `i` is paired with German line `i`. The dataset stores text pairs and tokenizes examples on demand. The collator then pads only to the longest source and target in the current batch.

Truncation retains `[EOS]`, which is important because it teaches the decoder when to stop generating.

In [3]:
class ParallelTextDataset(Dataset):
    def __init__(
        self,
        source_path,
        target_path,
        tokenizer,
        max_seq_len,
        max_examples=None,
    ):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.eos_token_id = tokenizer.token_to_id("[EOS]")

        pairs = []
        with (
            open(source_path, encoding="utf-8") as source_file,
            open(target_path, encoding="utf-8") as target_file,
        ):
            lines = zip_longest(source_file, target_file)
            if max_examples is not None:
                lines = islice(lines, max_examples)

            for source_line, target_line in lines:
                if source_line is None or target_line is None:
                    raise ValueError(
                        "Source and target files must contain the same number of lines"
                    )
                source_text = source_line.rstrip()
                target_text = target_line.rstrip()
                if source_text and target_text:
                    pairs.append((source_text, target_text))

        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def _encode(self, text):
        token_ids = self.tokenizer.encode(text).ids
        if len(token_ids) > self.max_seq_len:
            token_ids = token_ids[: self.max_seq_len - 1] + [self.eos_token_id]
        return token_ids

    def __getitem__(self, index):
        source_text, target_text = self.pairs[index]
        return {
            "source_ids": self._encode(source_text),
            "target_ids": self._encode(target_text),
        }


class TranslationCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def _pad(self, sequences):
        max_len = max(len(sequence) for sequence in sequences)
        padded = [
            sequence + [self.pad_token_id] * (max_len - len(sequence))
            for sequence in sequences
        ]
        return torch.tensor(padded, dtype=torch.long)

    def __call__(self, examples):
        source_ids = self._pad([example["source_ids"] for example in examples])
        target_ids = self._pad([example["target_ids"] for example in examples])

        decoder_input_ids = target_ids[:, :-1]
        labels = target_ids[:, 1:]

        return {
            "source_ids": source_ids,
            "decoder_input_ids": decoder_input_ids,
            "labels": labels,
            "source_padding_mask": source_ids.eq(self.pad_token_id),
            "decoder_padding_mask": decoder_input_ids.eq(self.pad_token_id),
        }

In [8]:
train_dataset = ParallelTextDataset(
    source_path=english_path,
    target_path=german_path,
    tokenizer=tokenizer,
    max_seq_len=context_length,
    max_examples=max_examples,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=TranslationCollator(pad_token_id),
)

batch = next(iter(train_loader))
for name, tensor in batch.items():
    print(f"{name:22s} {tuple(tensor.shape)}")

source_ids             (4, 11)
decoder_input_ids      (4, 12)
labels                 (4, 12)
source_padding_mask    (4, 11)
decoder_padding_mask   (4, 12)
